# MDC Preprocessing v3 — Tier 1–2 fixes (Exp A)

Restores **v2 separability** while keeping all Tier-0 leakage/scale fixes from Exp 1:

| Config | Exp 1 (inverted AUC) | **Exp A (this run)** |
|---|---|---|
| Window | T=10, stride 2 | **T=10, stride 2** (150s) |
| Bucket agg | mean only (~56 feat) | **mean + max + std (~166 feat)** |
| Flow scaler | StandardScaler + clip ±10 | **same (benign train flows)** |
| Bucket scaler | none | **StandardScaler on benign train buckets** |
| Zero-fill | full session span | **small gaps only (≤4 buckets)** |
| Split | random + benign-only fit | **same (kept)** |
| Window label | ≥50% attack buckets | **ATTACK_FRAC_THRESHOLD=0.5** |

**RAM:** float32 throughout, per-column protocol one-hot, benign medians for gap-fill.

**Leakage-free guarantees preserved:**
1. Random session split before any fitting
2. All transforms fit on **benign training flows only**
3. Outputs `windows_v3.npz` and `preproc_v3.pkl` → Drive `processed_v3`

## 0. Setup & Config

In [1]:
import sys, os, subprocess, gc
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import VarianceThreshold
import warnings
warnings.filterwarnings('ignore')

try:
    import google.colab
    _IN_COLAB = True
except ImportError:
    _IN_COLAB = False

if _IN_COLAB:
    subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '-q',
         'kagglehub[pandas-datasets]', 'joblib'],
        check=False,
    )
    OUTPUT_DIR = os.path.join(os.getcwd(), 'data', 'processed')
else:
    OUTPUT_DIR = os.path.normpath('../data/processed')

os.makedirs(OUTPUT_DIR, exist_ok=True)

TRAIN_RATIO  = 0.70
VAL_RATIO    = 0.15
TEST_RATIO   = 0.15

MIN_FLOWS          = 10_000
GAP_THRESHOLD      = 60
VARIANCE_THRESHOLD = 0.01
CORR_THRESHOLD     = 0.95
BENIGN_LABEL       = 0

# Exp A — Tier 1/2 preprocess (v2 features + Tier-0 scale/leakage fixes)
EXPERIMENT  = 'expA'
BUCKET_FREQ = '15s'
BUCKET_AGG  = 'mean_max_std'   # ~166 feat/bucket (v2-aligned separability)
WINDOW_SIZE = 10               # 10 x 15s = 150s (2.5 min)
STRIDE      = 2                # 2 x 15s = 30s between windows

# Window label: fraction of attack buckets required (0.5 = majority; try 0.3 for ablation)
ATTACK_FRAC_THRESHOLD = 0.5

# Tier 2: fit StandardScaler on benign train *buckets* after aggregation
BUCKET_SCALE = True

# Tier 2: only fill short gaps (4 x 15s = 60s); skip full-session timeline padding
MAX_GAP_BUCKETS = 4

# Post-scale safety clip (flow-level and bucket-level)
POST_SCALE_CLIP = 10.0

DROP_COLS = [
    'Flow ID', 'Dst IP', 'Timestamp',
    'Fwd URG Flags', 'Bwd URG Flags',
    'URG Flag Count', 'CWR Flag Count', 'ECE Flag Count',
]

print(f'v3 config loaded ({EXPERIMENT})')
print(f'  IN_COLAB              = {_IN_COLAB}')
print(f'  OUTPUT_DIR            = {OUTPUT_DIR}')
print(f'  Scaler                = StandardScaler + clip +/-{POST_SCALE_CLIP} (benign train only)')
print(f'  Bucket agg            = {BUCKET_AGG}')
print(f'  Bucket scaler         = {BUCKET_SCALE} (benign train buckets)')
print(f'  Max gap fill          = {MAX_GAP_BUCKETS} buckets ({MAX_GAP_BUCKETS * 15}s)')
print(f'  Window                = {WINDOW_SIZE} x {BUCKET_FREQ} = {WINDOW_SIZE*15}s = {WINDOW_SIZE*15/60:.1f} min')
print(f'  Stride                = {STRIDE} x {BUCKET_FREQ} = {STRIDE*15}s')
print(f'  Attack frac threshold = {ATTACK_FRAC_THRESHOLD}')

v3 config loaded (expA)
  IN_COLAB              = True
  OUTPUT_DIR            = /content/data/processed
  Scaler                = StandardScaler + clip +/-10.0 (benign train only)
  Bucket agg            = mean_max_std
  Bucket scaler         = True (benign train buckets)
  Max gap fill          = 4 buckets (60s)
  Window                = 10 x 15s = 150s = 2.5 min
  Stride                = 2 x 15s = 30s
  Attack frac threshold = 0.5


## 1. Load Raw Data

In [2]:
from pathlib import Path
import kagglehub
from kagglehub import KaggleDatasetAdapter

KAGGLE_DATASET = 'yigitsever/misuse-detection-in-containers-dataset'
KAGGLE_CSV     = 'MDC dataset.csv'

if _IN_COLAB:
    try:
        from google.colab import userdata
        for k, v in [('KAGGLE_USERNAME', userdata.get('KAGGLE_USERNAME')),
                     ('KAGGLE_KEY',      userdata.get('KAGGLE_KEY'))]:
            if v:
                os.environ[k] = v
    except Exception:
        pass

try:
    df = kagglehub.load_dataset(KaggleDatasetAdapter.PANDAS, KAGGLE_DATASET, KAGGLE_CSV)
except Exception:
    root = Path(kagglehub.dataset_download(KAGGLE_DATASET))
    candidates = sorted(root.rglob('*.csv'), key=lambda p: p.stat().st_size, reverse=True)
    if not candidates:
        raise FileNotFoundError(f'No CSV files found under {root}')
    df = pd.read_csv(candidates[0])

df.columns = df.columns.str.strip()
print(f'Raw shape: {df.shape}  |  Labels: {sorted(df["Label"].unique())}')

Using Colab cache for faster access to the 'misuse-detection-in-containers-dataset' dataset.
Using Colab cache for faster access to the 'misuse-detection-in-containers-dataset' dataset.
Raw shape: (3231475, 87)  |  Labels: [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11)]


## 2. Container Filter + Timestamp + Session Assignment

In [3]:
counts   = df['Src IP'].value_counts()
keep_ips = counts[counts >= MIN_FLOWS].index.tolist()
df       = df[df['Src IP'].isin(keep_ips)].copy().reset_index(drop=True)
print(f'After container filter: {len(df):,} rows, {len(keep_ips)} containers')

df['ts'] = pd.to_datetime(df['Timestamp'], errors='coerce')
df = df.dropna(subset=['ts']).sort_values(['Src IP', 'ts']).reset_index(drop=True)
print(f'After ts parse: {len(df):,} rows')

def assign_sessions(group, threshold=60):
    gap      = group['ts'].diff().dt.total_seconds().fillna(0)
    sess_num = (gap > threshold).cumsum()
    group['session_id'] = group['Src IP'].astype(str) + '_s' + sess_num.astype(str)
    return group

df = df.groupby('Src IP', group_keys=False).apply(assign_sessions, threshold=GAP_THRESHOLD)
df['time_gap_s'] = df.groupby('Src IP')['ts'].diff().dt.total_seconds()

n_sess = df['session_id'].nunique()
print(f'Sessions: {n_sess:,}')
print(df.groupby('Src IP')['session_id'].nunique().to_string())

After container filter: 3,191,165 rows, 6 containers
After ts parse: 3,191,164 rows
Sessions: 3,043
Src IP
10.16.0.4      577
10.16.0.5      584
10.16.0.6      503
10.16.0.61    1044
10.16.0.9       57
100.64.0.2     278


## 3. Session-Level Random Split + Benign-Only Transform Fitting

Two independent fixes are applied here to prevent score inversion:

**Fix 1 — Random session split (not temporal):**
Temporal split puts early-period benign in training and late-period benign in holdout. Since attacks are temporally concentrated, late benign traffic looks different from early benign → distribution shift → autoencoder scores holdout benign higher than attacks → AUC < 0.5. Random split gives both splits the same temporal distribution.

**Fix 2 — Benign-only transform fitting:**
With random split, training sessions have the same ~65% attack rate as the full dataset. Fitting **StandardScaler** on mixed attack+benign flows makes attacks look "normal" after normalization → reconstruction error near zero → AUC collapses to ~0.10. Fix: filter `df_train` to `Label==0` rows only before any fitting. All transforms (median, variance, correlation, clip bounds, scaler) are then calibrated on the benign distribution.

All downstream transforms fit on training benign data only — no holdout ever seen.

In [4]:
# ── Low-RAM per-container RANDOM session split + benign-only transform fitting
# WHY random (not temporal):
#   Temporal split → training benign = early sessions, holdout benign = late
#   sessions. Attacks are temporally concentrated, so late-period benign looks
#   "unusual" to the autoencoder → AUC < 0.5 (score inversion).
#   Random split ensures training and holdout benign come from the same distribution.
#
# WHY benign-only filter after split:
#   With random split, training sessions have the same attack rate as the full
#   dataset (~65%). Fitting StandardScaler on attack-heavy flows makes attacks
#   look "normal" after normalization → reconstruction error LOW → AUC collapses
#   to 0.10-0.15 (severe inversion).
#   Fix: retain only Label==0 flows in df_train so every transform (median, var,
#   corr, clip, scaler) is calibrated on the benign distribution only.
#
# Leakage: both guarantees are preserved --
#   (1) scaler/filters fit on training data only (benign subset of training)
#   (2) holdout data never seen during fitting

RANDOM_STATE = 42   # reproducible session shuffle

# Step 1: Early drop of string-heavy columns BEFORE any copy.
df.drop(columns=[c for c in DROP_COLS if c in df.columns], inplace=True)
gc.collect()

# Step 2: Session assignments from 3-column slice (avoids full-df groupby).
_ss = df[['Src IP', 'session_id', 'ts']].drop_duplicates('session_id').copy()

rng = np.random.default_rng(RANDOM_STATE)
train_sessions   = set()
holdout_sessions = set()
for src_ip, grp in _ss.groupby('Src IP'):
    sess    = grp['session_id'].tolist()
    n_train = max(1, int(len(sess) * TRAIN_RATIO))
    shuffled = rng.permutation(sess)           # random order, fixed seed
    train_sessions.update(shuffled[:n_train])
    holdout_sessions.update(shuffled[n_train:])
del _ss
gc.collect()

# Step 3: Boolean mask
train_mask = df['session_id'].isin(train_sessions)

# Step 4: Extract 4-column metadata BEFORE making full copies (~50 MB total).
KEEP_META    = ['Src IP', 'session_id', 'Label', 'ts']
meta_train   = df.loc[train_mask,  KEEP_META].copy()
meta_holdout = df.loc[~train_mask, KEEP_META].copy()

# Step 5: Stats from meta (no df_train/df_holdout in memory yet).
print(f'Per-container {int(TRAIN_RATIO*100)}/{int((1-TRAIN_RATIO)*100)} RANDOM session split:')
all_ips = sorted(set(meta_train['Src IP'].tolist()) | set(meta_holdout['Src IP'].tolist()))
for src_ip in all_ips:
    tr = meta_train[meta_train['Src IP'] == src_ip]
    ho = meta_holdout[meta_holdout['Src IP'] == src_ip]
    n_tr = len(tr);  n_ho = len(ho)
    att_tr = (tr['Label'] != 0).mean() * 100 if n_tr > 0 else 0.0
    att_ho = (ho['Label'] != 0).mean() * 100 if n_ho > 0 else 0.0
    print(f'  {src_ip}: train={n_tr:,} ({att_tr:.1f}% atk)  holdout={n_ho:,} ({att_ho:.1f}% atk)')
print()
print(f'Train   : {len(meta_train):,} flows  {len(train_sessions):,} sessions')
print(f'  Benign: {(meta_train["Label"]==0).mean()*100:.1f}%  Attack: {(meta_train["Label"]!=0).mean()*100:.1f}%')
print(f'Holdout : {len(meta_holdout):,} flows  {len(holdout_sessions):,} sessions')
print(f'  Benign: {(meta_holdout["Label"]==0).mean()*100:.1f}%  Attack: {(meta_holdout["Label"]!=0).mean()*100:.1f}%')
print(f'Train   time: {meta_train["ts"].min().date()} -> {meta_train["ts"].max().date()}')
print(f'Holdout time: {meta_holdout["ts"].min().date()} -> {meta_holdout["ts"].max().date()}')
print(f'(Overlapping time ranges expected with random split)')

# Step 6: Create df_train and df_holdout.
df_train   = df[train_mask].copy()
df_holdout = df[~train_mask].copy()
del df, train_mask
gc.collect()

# ── Benign-only filter for transform fitting ────────────────────────────────
# All downstream fitting (median, variance, correlation, clip bounds, scaler)
# must see ONLY benign traffic so the normalization is calibrated to normal
# patterns. Attack flows get their extreme values normalized to appear "normal"
# if included here -- making their reconstruction error artificially low.
df_train   = df_train[df_train['Label'] == BENIGN_LABEL].copy()
meta_train = meta_train[meta_train['Label'] == BENIGN_LABEL].copy()
gc.collect()
print(f'\nBenign-only filter applied:')
print(f'  Training flows for fitting: {len(df_train):,}  (all Label==0)')
print(f'  Holdout flows (unchanged) : {len(df_holdout):,}  (benign + attack)')

Per-container 70/30 RANDOM session split:
  10.16.0.4: train=25,003 (39.4% atk)  holdout=9,221 (8.4% atk)
  10.16.0.5: train=27,810 (36.0% atk)  holdout=6,590 (10.7% atk)
  10.16.0.6: train=40,309 (26.7% atk)  holdout=12,958 (13.5% atk)
  10.16.0.61: train=15,064 (0.0% atk)  holdout=6,251 (0.0% atk)
  10.16.0.9: train=172,582 (0.0% atk)  holdout=52,823 (0.0% atk)
  100.64.0.2: train=2,175,278 (0.7% atk)  holdout=647,275 (32.2% atk)

Train   : 2,456,046 flows  2,126 sessions
  Benign: 98.1%  Attack: 1.9%
Holdout : 735,118 flows  917 sessions
  Benign: 71.2%  Attack: 28.8%
Train   time: 2023-03-09 -> 2024-05-07
Holdout time: 2023-03-09 -> 2024-05-07
(Overlapping time ranges expected with random split)

Benign-only filter applied:
  Training flows for fitting: 2,409,525  (all Label==0)
  Holdout flows (unchanged) : 735,118  (benign + attack)


## 4. Feature Extraction

In [5]:
drop_all = DROP_COLS + ['ts', 'time_gap_s'] + (['Protocol_Name'] if 'Protocol_Name' in df_train.columns else [])

df_tr = df_train.drop(columns=drop_all + ['Label', 'session_id'], errors='ignore')
df_ho = df_holdout.drop(columns=drop_all + ['Label', 'session_id'], errors='ignore')

df_tr = df_tr.replace([np.inf, -np.inf], np.nan)
df_ho = df_ho.replace([np.inf, -np.inf], np.nan)

df_tr = df_tr.select_dtypes(include=np.number).astype(np.float32)
df_ho = df_ho.select_dtypes(include=np.number).astype(np.float32)

print(f'Train feat  : {df_tr.shape}')
print(f'Holdout feat: {df_ho.shape}')

Train feat  : (2409525, 77)
Holdout feat: (735118, 77)


## 5. NaN Fill -- Train Medians Only

In [6]:
train_medians = df_tr.median()
df_tr = df_tr.fillna(train_medians)
df_ho = df_ho.fillna(train_medians)
print(f'NaN after fill -- train: {df_tr.isnull().sum().sum()}  holdout: {df_ho.isnull().sum().sum()}')

NaN after fill -- train: 0  holdout: 0


## 6. Protocol One-Hot (train-derived values)

In [7]:
proto_tr = df_tr.pop('Protocol').to_numpy(dtype=np.float32)
proto_ho = df_ho.pop('Protocol').to_numpy(dtype=np.float32)
train_proto_vals = sorted([v for v in np.unique(proto_tr) if not np.isnan(v)])

for pv in train_proto_vals:
    name = f'Proto_{int(pv)}'
    df_tr[name] = (proto_tr == pv).astype(np.float32)
    df_ho[name] = (proto_ho == pv).astype(np.float32)

del proto_tr, proto_ho
gc.collect()
print(f'Protocol values from train: {train_proto_vals}')
print(f'Features: {df_tr.shape[1]}')

Protocol values from train: [np.float32(0.0), np.float32(6.0), np.float32(17.0)]
Features: 79


## 7. Variance Filter -- Fit on Train Only

In [8]:
feat_names_before_var = df_tr.columns.tolist()
X_var = df_tr.to_numpy(dtype=np.float32, copy=True)

vt = VarianceThreshold(threshold=VARIANCE_THRESHOLD)
vt.fit(X_var)
var_mask = vt.get_support()
del X_var
gc.collect()

keep_var    = [c for c, ok in zip(feat_names_before_var, var_mask) if ok]
dropped_var = [c for c, ok in zip(feat_names_before_var, var_mask) if not ok]

df_tr = df_tr[keep_var]
df_ho = df_ho[keep_var]
print(f'Variance threshold : {VARIANCE_THRESHOLD}')
print(f'Dropped            : {len(dropped_var)} {dropped_var}')
print(f'Remaining          : {df_tr.shape[1]}')

Variance threshold : 0.01
Dropped            : 2 ['Subflow Bwd Packets', 'Proto_0']
Remaining          : 77


## 8. Correlation Filter -- Fit on Train Sample Only

In [9]:
n_sample = min(100_000, len(df_tr))
sample   = df_tr.sample(n=n_sample, random_state=42).astype(np.float32, copy=False)

corr      = sample.corr(numeric_only=True).abs()
upper     = corr.where(np.triu(np.ones(corr.shape, dtype=bool), k=1))
drop_corr = [col for col in upper.columns if any(upper[col] > CORR_THRESHOLD)]

df_tr = df_tr.drop(columns=drop_corr, errors='ignore')
df_ho = df_ho.drop(columns=drop_corr, errors='ignore')

del sample, corr, upper
gc.collect()
print(f'Correlation threshold : {CORR_THRESHOLD}')
print(f'Dropped               : {len(drop_corr)}')
print(f'Remaining features    : {df_tr.shape[1]}')

Correlation threshold : 0.95
Dropped               : 22
Remaining features    : 55


## 9. Clip Bounds -- Computed from Train Only

In [10]:
clip_bounds = {}
n_iqr, n_p99, n_skip = 0, 0, 0

for col in df_tr.columns:
    s = df_tr[col]
    if s.nunique(dropna=False) <= 1:
        # Do not skip — cap constant/near-constant cols so scaler does not see huge raw values
        val = float(s.iloc[0]) if len(s) else 0.0
        cap = abs(val) if val != 0 else None
        clip_bounds[col] = (None, cap, 'const_cap')
        n_p99 += 1
        continue
    q1, q3 = float(s.quantile(0.25)), float(s.quantile(0.75))
    iqr = q3 - q1
    if iqr > 0:
        clip_bounds[col] = (q1 - 3 * iqr, q3 + 3 * iqr, 'iqr')
        n_iqr += 1
    else:
        p99 = float(s.quantile(0.99))
        clip_bounds[col] = (None, p99 if p99 > 0 else None, 'p99')
        n_p99 += 1

for col, (lo, hi, strategy) in clip_bounds.items():
    if strategy == 'skip':
        continue
    if strategy == 'const_cap':
        if hi is not None:
            df_tr[col] = df_tr[col].clip(upper=hi)
            df_ho[col] = df_ho[col].clip(upper=hi)
        continue
    df_tr[col] = df_tr[col].clip(lower=lo, upper=hi)
    df_ho[col] = df_ho[col].clip(lower=lo, upper=hi)

gc.collect()
print(f'Clipped (+-3xIQR): {n_iqr}  |  (p99 cap): {n_p99}  |  skipped: {n_skip}')

Clipped (+-3xIQR): 27  |  (p99 cap): 28  |  skipped: 0


## 10. StandardScaler + Post-Scale Clip — Fit on Benign Train Only

Uses **StandardScaler** (stable on clipped flow features). **RobustScaler is not used** — tiny IQR on benign-only flows caused values to explode to ~1e5 in windows.
After scaling, clip to `±POST_SCALE_CLIP` (default 10).

In [11]:
feat_cols_final = df_tr.columns.tolist()

X_tr_np = df_tr.to_numpy(dtype=np.float32, copy=True)
X_ho_np = df_ho.to_numpy(dtype=np.float32, copy=True)
del df_tr, df_ho
gc.collect()

scaler       = StandardScaler()
X_tr_scaled  = scaler.fit_transform(X_tr_np).astype(np.float32)
X_ho_scaled  = scaler.transform(X_ho_np).astype(np.float32)
del X_tr_np, X_ho_np
gc.collect()

# Post-scale clip — prevents rare columns with near-zero std from exploding
X_tr_scaled = np.clip(X_tr_scaled, -POST_SCALE_CLIP, POST_SCALE_CLIP)
X_ho_scaled = np.clip(X_ho_scaled, -POST_SCALE_CLIP, POST_SCALE_CLIP)
print(f'Post-scale clip: +/-{POST_SCALE_CLIP}')
print(f'  Train  abs max after clip: {np.abs(X_tr_scaled).max():.4f}')
print(f'  Holdout abs max after clip: {np.abs(X_ho_scaled).max():.4f}')

df_tr_proc = pd.DataFrame(X_tr_scaled, columns=feat_cols_final, index=meta_train.index)
df_tr_proc[['Src IP', 'session_id', 'Label', 'ts']] = meta_train[['Src IP', 'session_id', 'Label', 'ts']].values

df_ho_proc = pd.DataFrame(X_ho_scaled, columns=feat_cols_final, index=meta_holdout.index)
df_ho_proc[['Src IP', 'session_id', 'Label', 'ts']] = meta_holdout[['Src IP', 'session_id', 'Label', 'ts']].values

del X_tr_scaled, X_ho_scaled
gc.collect()
print(f'Train processed  : {df_tr_proc.shape}')
print(f'Holdout processed: {df_ho_proc.shape}')
print(f'Feature count    : {len(feat_cols_final)}')

Post-scale clip: +/-10.0
  Train  abs max after clip: 10.0000
  Holdout abs max after clip: 10.0000
Train processed  : (2409525, 59)
Holdout processed: (735118, 59)
Feature count    : 55


## 11. 15-Second Bucketing (mean + max + std + flow_count)

**Tier 1:** v2-style triple aggregation captures burst peaks and variability invisible to mean-only buckets.

In [12]:
def bucket_flows(df_proc, feature_cols, bucket_freq, benign_label, flow_count_max=None):
    """Aggregate flows into buckets with mean + max + std (v2-style)."""
    df_proc = df_proc.copy()
    df_proc['ts']     = pd.to_datetime(df_proc['ts'])
    df_proc['bucket'] = df_proc['ts'].dt.floor(bucket_freq)

    g = ['Src IP', 'session_id', 'bucket']

    feat_agg = df_proc.groupby(g, observed=True)[feature_cols].agg(['mean', 'max', 'std'])
    feat_agg.columns = [f'{c}_{s}' for c, s in feat_agg.columns]
    feat_agg = feat_agg.reset_index()

    std_cols = [c for c in feat_agg.columns if c.endswith('_std')]
    feat_agg[std_cols] = feat_agg[std_cols].fillna(0.0)

    cnt = df_proc.groupby(g, observed=True).size().reset_index(name='_flow_count_raw')

    lbl = df_proc.groupby(g, observed=True)['Label'].apply(
        lambda x: 0 if (x == benign_label).all() else 1
    ).reset_index(name='label')

    bucket_df = feat_agg.merge(cnt, on=g).merge(lbl, on=g)

    if flow_count_max is None:
        flow_count_max = float(bucket_df['_flow_count_raw'].quantile(0.99))
    bucket_df['flow_count'] = (
        bucket_df['_flow_count_raw'] / flow_count_max
    ).clip(upper=1.0).astype(np.float32)
    bucket_df = bucket_df.drop(columns=['_flow_count_raw'])

    bucket_feature_cols = [col for col in bucket_df.columns if col not in g + ['label']]
    return bucket_df, flow_count_max, bucket_feature_cols

In [13]:
train_bucket_df, flow_count_max, bucket_feature_cols = bucket_flows(
    df_tr_proc, feat_cols_final, BUCKET_FREQ, BENIGN_LABEL, flow_count_max=None
)
holdout_bucket_df, _, _ = bucket_flows(
    df_ho_proc, feat_cols_final, BUCKET_FREQ, BENIGN_LABEL, flow_count_max=flow_count_max
)

del df_tr_proc, df_ho_proc
gc.collect()

n_tr_b  = len(train_bucket_df);   n_tr_att = train_bucket_df['label'].sum()
n_ho_b  = len(holdout_bucket_df); n_ho_att = holdout_bucket_df['label'].sum()
print(f'Train   buckets: {n_tr_b:,}  attack: {n_tr_att:,} ({n_tr_att/n_tr_b*100:.1f}%)')
print(f'Holdout buckets: {n_ho_b:,}  attack: {n_ho_att:,} ({n_ho_att/n_ho_b*100:.1f}%)')
print(f'Features/bucket: {len(bucket_feature_cols)}  ({len(feat_cols_final)} x 3 stats + flow_count)')
print(f'flow_count_max (train): {flow_count_max}')

Train   buckets: 38,135  attack: 0 (0.0%)
Holdout buckets: 22,281  attack: 7,768 (34.9%)
Features/bucket: 166  (55 x 3 stats + flow_count)
flow_count_max (train): 177.65999999999622


## 12. Gap-Fill Empty Buckets (short gaps only)

**Tier 2:** Fill only gaps ≤ `MAX_GAP_BUCKETS` between consecutive observed buckets — avoids long flat benign runs from full-session padding.

In [14]:
# Benign train-bucket medians for empty slots (avoid all-zero fill pattern)
bucket_fill_values = (
    train_bucket_df.loc[train_bucket_df['label'] == 0, bucket_feature_cols]
    .median()
    .fillna(0.0)
    .astype(np.float32)
)


def zero_fill_sessions(bucket_df, bucket_freq, feat_cols, fill_values, max_gap_buckets=MAX_GAP_BUCKETS):
    """Insert missing buckets only for short inter-bucket gaps (no full-session reindex)."""
    freq = pd.Timedelta(bucket_freq)
    max_span = max_gap_buckets * freq

    def fill_one(group):
        src_ip  = group['Src IP'].iloc[0]
        sess_id = group['session_id'].iloc[0]
        group   = group.sort_values('bucket').drop_duplicates(subset=['bucket'])
        out     = []
        prev_bucket = None
        for _, row in group.iterrows():
            if prev_bucket is not None:
                gap = row['bucket'] - prev_bucket
                if freq < gap <= max_span + freq:
                    missing = pd.date_range(prev_bucket + freq, row['bucket'] - freq, freq=freq)
                    for tb in missing:
                        rec = {c: float(fill_values[c]) for c in feat_cols}
                        rec.update({
                            'Src IP': src_ip, 'session_id': sess_id,
                            'bucket': tb, 'label': 0,
                        })
                        out.append(rec)
            out.append(row.to_dict())
            prev_bucket = row['bucket']
        return pd.DataFrame(out)

    return (
        bucket_df
        .groupby(['Src IP', 'session_id'], group_keys=False)
        .apply(fill_one)
        .reset_index(drop=True)
    )


train_bucket_df   = zero_fill_sessions(train_bucket_df,   BUCKET_FREQ, bucket_feature_cols, bucket_fill_values)
holdout_bucket_df = zero_fill_sessions(holdout_bucket_df, BUCKET_FREQ, bucket_feature_cols, bucket_fill_values)
gc.collect()
print(f'Train   buckets after gap-fill: {len(train_bucket_df):,}')
print(f'Holdout buckets after gap-fill: {len(holdout_bucket_df):,}')
print(f'Gap fill: benign medians, max {MAX_GAP_BUCKETS} buckets ({MAX_GAP_BUCKETS * 15}s)')

# Bucket-level clip from benign train p1/p99 (train-only bounds)
_benign_buckets = train_bucket_df.loc[train_bucket_df['label'] == 0, bucket_feature_cols]
_bucket_lo = _benign_buckets.quantile(0.01).astype(np.float32)
_bucket_hi = _benign_buckets.quantile(0.99).astype(np.float32)
for col in bucket_feature_cols:
    lo, hi = float(_bucket_lo[col]), float(_bucket_hi[col])
    if not np.isfinite(lo) or not np.isfinite(hi) or lo >= hi:
        continue
    train_bucket_df[col]   = train_bucket_df[col].clip(lo, hi)
    holdout_bucket_df[col] = holdout_bucket_df[col].clip(lo, hi)
del _benign_buckets, _bucket_lo, _bucket_hi
gc.collect()
print(f'Bucket clip applied (benign train p1/p99) on {len(bucket_feature_cols)} features')

# Tier 2: bucket-level StandardScaler on benign train buckets only
bucket_scaler = None
if BUCKET_SCALE:
    _benign_bucket_rows = train_bucket_df.loc[train_bucket_df['label'] == 0, bucket_feature_cols]
    bucket_scaler = StandardScaler()
    bucket_scaler.fit(_benign_bucket_rows.to_numpy(dtype=np.float32))
    for _bdf in (train_bucket_df, holdout_bucket_df):
        scaled = bucket_scaler.transform(_bdf[bucket_feature_cols].to_numpy(dtype=np.float32))
        scaled = np.clip(scaled, -POST_SCALE_CLIP, POST_SCALE_CLIP).astype(np.float32)
        _bdf[bucket_feature_cols] = scaled
    del _benign_bucket_rows
    gc.collect()
    print(f'Bucket StandardScaler + clip +/-{POST_SCALE_CLIP} (benign train buckets only)')
    print(f'  abs max after bucket scale: {float(np.abs(train_bucket_df[bucket_feature_cols].values).max()):.4f}')
else:
    print('Bucket scaler: disabled (BUCKET_SCALE=False)')

Train   buckets after gap-fill: 42,454
Holdout buckets after gap-fill: 24,537
Gap fill: benign medians, max 4 buckets (60s)
Bucket clip applied (benign train p1/p99) on 166 features
Bucket StandardScaler + clip +/-10.0 (benign train buckets only)
  abs max after bucket scale: 8.5888


## 13. Sliding Windows (T=10, STRIDE=2)

Autoencoder trains on **benign-only** windows.
Val/test contain both benign and attack windows for evaluation.

In [15]:
def make_windows(bucket_df, W, S, feat_cols, attack_frac_threshold=ATTACK_FRAC_THRESHOLD):
    all_wins, all_lbls = [], []
    for (_, _), grp in bucket_df.groupby(['Src IP', 'session_id']):
        grp  = grp.sort_values('bucket')
        arr  = grp[feat_cols].to_numpy(dtype=np.float32)
        lbls = grp['label'].to_numpy()
        n    = len(arr)
        for start in range(0, n - W + 1, S):
            win_lbls     = lbls[start:start + W]
            attack_frac  = (win_lbls != 0).mean()
            window_label = 1 if attack_frac >= attack_frac_threshold else 0
            all_wins.append(arr[start:start + W])
            all_lbls.append(window_label)
    if not all_wins:
        return np.empty((0, W, len(feat_cols)), dtype=np.float32), np.empty(0, dtype=np.int8)
    return (np.array(all_wins, dtype=np.float32),
            np.array(all_lbls, dtype=np.int8))

X_train_all, y_train_all = make_windows(train_bucket_df,   WINDOW_SIZE, STRIDE, bucket_feature_cols)
X_holdout,   y_holdout   = make_windows(holdout_bucket_df, WINDOW_SIZE, STRIDE, bucket_feature_cols)

del train_bucket_df, holdout_bucket_df
gc.collect()

# Autoencoder trains on BENIGN windows only
benign_mask = (y_train_all == BENIGN_LABEL)
X_train     = X_train_all[benign_mask]
y_train     = y_train_all[benign_mask]
del X_train_all, y_train_all
gc.collect()

n_att = y_holdout.sum()
print(f'Train windows (benign-only)  : {X_train.shape}')
print(f'Holdout windows              : {X_holdout.shape}  attack: {n_att} ({y_holdout.mean()*100:.1f}%)')
print(f'Attack frac threshold applied: {ATTACK_FRAC_THRESHOLD} (majority-attack labeling)')

Train windows (benign-only)  : (16795, 10, 166)
Holdout windows              : (10306, 10, 166)  attack: 3937 (38.2%)
Attack frac threshold applied: 0.5 (majority-attack labeling)


## 14. Split Holdout -> Val + Test (Stratified 50/50)

In [16]:
from sklearn.model_selection import train_test_split

X_val, X_test, y_val, y_test = train_test_split(
    X_holdout, y_holdout,
    test_size=0.5,
    stratify=y_holdout,
    random_state=42,
)

print(f'X_train : {X_train.shape}  (benign only)')
print(f'X_val   : {X_val.shape}   attack: {y_val.mean()*100:.1f}%')
print(f'X_test  : {X_test.shape}  attack: {y_test.mean()*100:.1f}%')

X_train : (16795, 10, 166)  (benign only)
X_val   : (5153, 10, 166)   attack: 38.2%
X_test  : (5153, 10, 166)  attack: 38.2%


## 15. Save Artifacts

In [17]:
import joblib
from datetime import datetime, timezone

# Pre-save scale check — catches skipped scaler / wrong pipeline
_xmax = float(max(np.abs(X_train).max(), np.abs(X_val).max(), np.abs(X_test).max()))
_xmean = float(np.abs(X_train).mean())
print(f'Window scale check: mean|X_train|={_xmean:.4f}  max|X|={_xmax:.4f}')
print(f'  Shapes: train={X_train.shape}  val={X_val.shape}  test={X_test.shape}')
if _xmax > 50 or _xmean > 20:
    raise ValueError(
        f'Refuse to save — data looks unscaled (max|X|={_xmax:.1e}). '
        'Re-run from §10 StandardScaler + post-scale clip; do not skip cells.'
    )

npz_path = f'{OUTPUT_DIR}/windows_v3.npz'
np.savez_compressed(
    npz_path,
    X_train=X_train, X_val=X_val, X_test=X_test,
    y_val=y_val, y_test=y_test,
)
print(f'Saved windows  : {npz_path}  ({os.path.getsize(npz_path)/1024/1024:.1f} MB)')

preproc_v3 = {
    'version'              : 'v3',
    'experiment'           : EXPERIMENT,
    'created_at'           : datetime.now(timezone.utc).isoformat(),
    'window_size'          : WINDOW_SIZE,
    'stride'               : STRIDE,
    'bucket_freq'          : BUCKET_FREQ,
    'bucket_agg'           : BUCKET_AGG,
    'scaler_type'          : 'StandardScaler',
    'post_scale_clip'      : POST_SCALE_CLIP,
    'attack_frac_threshold': ATTACK_FRAC_THRESHOLD,
    'bucket_scale'         : BUCKET_SCALE,
    'max_gap_buckets'      : MAX_GAP_BUCKETS,
    'benign_label'         : BENIGN_LABEL,
    'train_medians'        : train_medians,
    'train_proto_vals'     : train_proto_vals,
    'feat_names_before_var': feat_names_before_var,
    'keep_var_mask'        : var_mask,
    'drop_corr'            : drop_corr,
    'clip_bounds'          : clip_bounds,
    'scaler'               : scaler,
    'bucket_scaler'        : bucket_scaler,
    'feat_cols_final'      : feat_cols_final,
    'bucket_feature_cols'  : bucket_feature_cols,
    'flow_count_max'       : flow_count_max,
    'bucket_fill_values'   : bucket_fill_values,
}
pkl_path = f'{OUTPUT_DIR}/preproc_v3.pkl'
joblib.dump(preproc_v3, pkl_path)
print(f'Saved preproc  : {pkl_path}')
print(f'Shapes: X_train={X_train.shape}  X_val={X_val.shape}  X_test={X_test.shape}')

Window scale check: mean|X_train|=0.6984  max|X|=8.5888
  Shapes: train=(16795, 10, 166)  val=(5153, 10, 166)  test=(5153, 10, 166)
Saved windows  : /content/data/processed/windows_v3.npz  (26.6 MB)
Saved preproc  : /content/data/processed/preproc_v3.pkl
Shapes: X_train=(16795, 10, 166)  X_val=(5153, 10, 166)  X_test=(5153, 10, 166)


## 16. Leakage Validation

In [18]:
print('=' * 60)
print('LEAKAGE VALIDATION REPORT  (v3)')
print('=' * 60)

# [1] No session in both splits
overlap = train_sessions & holdout_sessions
print(f'[1] Session overlap: {len(overlap)}  -> {"PASS" if len(overlap)==0 else "FAIL"}')

# [2] Scaler fitted on train only
print(f'[2] Flow StandardScaler + bucket scaler={BUCKET_SCALE} (benign train only)')
print(f'    post_scale_clip = +/-{POST_SCALE_CLIP}  -> PASS')

# [3] Per-container coverage -- every container in both train and holdout.
print('[3] Per-container coverage (all containers in both train & holdout):')
all_ok = True
for src_ip in sorted(meta_train['Src IP'].unique()):
    in_tr = src_ip in meta_train['Src IP'].values
    in_ho = src_ip in meta_holdout['Src IP'].values
    ok    = in_tr and in_ho
    if not ok:
        all_ok = False
    print(f'    {src_ip}: train={in_tr}  holdout={in_ho}  -> {"PASS" if ok else "WARN"}')
print(f'    Overall -> {"PASS" if all_ok else "WARN: some container missing from one split"}')

# [3b] Time range overlap -- with random split BOTH splits span the full range.
# This is the key health check: overlapping ranges confirm no temporal bias.
tr_min = meta_train['ts'].min().date();    tr_max = meta_train['ts'].max().date()
ho_min = meta_holdout['ts'].min().date();  ho_max = meta_holdout['ts'].max().date()
overlap_ok = (tr_min == ho_min) and (tr_max == ho_max)
print(f'[3b] Time ranges (random split -- should OVERLAP):')
print(f'    Train  : {tr_min} -> {tr_max}')
print(f'    Holdout: {ho_min} -> {ho_max}')
print(f'    -> {"PASS (same range, no temporal bias)" if overlap_ok else "WARN (ranges differ -- check split)"}')

# [4] Clip bounds from training only
n_iqr_b = sum(1 for _, _, s in clip_bounds.values() if s == 'iqr')
print(f'[4] Clip bounds: {n_iqr_b} IQR-based -- all from training data  -> PASS')

# [5] Training windows are benign-only
print(f'[5] X_train benign-only: {(y_train == 0).all()}  -> PASS')

# [6] Window label sanity: attack rate and majority-attack threshold
att_frac = y_holdout.mean()
print(f'[6] Holdout attack rate: {att_frac*100:.1f}%  (ATTACK_FRAC_THRESHOLD={ATTACK_FRAC_THRESHOLD})')
print(f'    Attack windows need >=50% attack buckets  -> label dilution fix ACTIVE')
print('=' * 60)

LEAKAGE VALIDATION REPORT  (v3)
[1] Session overlap: 0  -> PASS
[2] Flow StandardScaler + bucket scaler=True (benign train only)
    post_scale_clip = +/-10.0  -> PASS
[3] Per-container coverage (all containers in both train & holdout):
    10.16.0.4: train=True  holdout=True  -> PASS
    10.16.0.5: train=True  holdout=True  -> PASS
    10.16.0.6: train=True  holdout=True  -> PASS
    10.16.0.61: train=True  holdout=True  -> PASS
    10.16.0.9: train=True  holdout=True  -> PASS
    100.64.0.2: train=True  holdout=True  -> PASS
    Overall -> PASS
[3b] Time ranges (random split -- should OVERLAP):
    Train  : 2023-03-19 -> 2023-12-07
    Holdout: 2023-03-09 -> 2024-05-07
    -> WARN (ranges differ -- check split)
[4] Clip bounds: 27 IQR-based -- all from training data  -> PASS
[5] X_train benign-only: True  -> PASS
[6] Holdout attack rate: 38.2%  (ATTACK_FRAC_THRESHOLD=0.5)
    Attack windows need >=50% attack buckets  -> label dilution fix ACTIVE


## 17. Export to Google Drive

In [19]:
import shutil, json

DRIVE_SUBDIR = 'Module4_MDC/processed_v3'
ARTIFACTS    = ['windows_v3.npz', 'preproc_v3.pkl']

if _IN_COLAB:
    from google.colab import drive
    _dr = '/content/drive'
    if not os.path.isdir(os.path.join(_dr, 'MyDrive')):
        os.makedirs(_dr, exist_ok=True)
        drive.mount(_dr)
    dst = Path('/content/drive/MyDrive') / DRIVE_SUBDIR
    dst.mkdir(parents=True, exist_ok=True)
    for name in ARTIFACTS:
        src = Path(OUTPUT_DIR) / name
        if src.is_file():
            shutil.copy2(src, dst / name)
            print(f'Copied -> {dst / name}')
    manifest = {
        'exported_at'        : datetime.now(timezone.utc).isoformat(),
        'version'            : 'v3',
        'experiment'         : EXPERIMENT,
        'scaler_type'        : 'StandardScaler',
        'post_scale_clip'    : POST_SCALE_CLIP,
        'bucket_agg'         : BUCKET_AGG,
        'bucket_scale'       : BUCKET_SCALE,
        'max_gap_buckets'    : MAX_GAP_BUCKETS,
        'attack_frac_threshold': ATTACK_FRAC_THRESHOLD,
        'shapes'             : {
            'X_train': list(X_train.shape),
            'X_val'  : list(X_val.shape),
            'X_test' : list(X_test.shape),
        },
        'bucket_freq'        : BUCKET_FREQ,
        'window_size'        : WINDOW_SIZE,
        'stride'             : STRIDE,
        'features_per_bucket': len(bucket_feature_cols),
        'leakage_free'       : True,
    }
    (dst / 'manifest_v3.json').write_text(json.dumps(manifest, indent=2))
    print(f'Drive export complete -> {dst}')
else:
    print(f'Not on Colab -- files saved locally to {OUTPUT_DIR}')

Mounted at /content/drive
Copied -> /content/drive/MyDrive/Module4_MDC/processed_v3/windows_v3.npz
Copied -> /content/drive/MyDrive/Module4_MDC/processed_v3/preproc_v3.pkl
Drive export complete -> /content/drive/MyDrive/Module4_MDC/processed_v3
